# ShadowTagAi Vertex AI Initialization

**Comprehensive setup for ShadowTagAi Intelligence Pipeline infrastructure**

## Components:
- Vertex AI & GCS setup
- Gemini models (Pro, Flash, Image Generation)
- ShadowTagAi agent prompts (swiper, verdict, vcm, geos, odor, tokable)
- OCR & vision tools
- Monte Carlo valuation engine
- Specialized workflows

**Usage**: Run cells sequentially to initialize full ShadowTagAi stack

## 1. Base Dependencies

In [ ]:
%%bash
python -V
pip -q install --upgrade google-cloud-aiplatform google-cloud-storage google-cloud-vision google-auth google-auth-oauthlib Pillow numpy pandas matplotlib

## 2. Vertex AI & GCS Initialization

In [ ]:
%%python
import google.cloud.aiplatform as a
from google.cloud import storage as gs

# Configuration
p = "YOUR_GCP_PROJECT"  # Replace with your GCP project ID
r = "us-central1"  # Region
b = "shadowtagai-bucket"  # GCS bucket name

# Initialize
a.init(project=p, location=r)
c = gs.Client(project=p)


# Helper functions for GCS
def up(local, blob):
    """Upload local file to GCS"""
    c.bucket(b).blob(blob).upload_from_filename(local)


def dl(blob, local):
    """Download GCS blob to local file"""
    c.bucket(b).blob(blob).download_to_filename(local)


print(f"✓ Vertex AI initialized (project={p}, region={r})")
print(f"✓ GCS bucket: {b}")

## 3. Gemini Model Initialization

In [ ]:
%%python
from vertexai import init as vinit
from vertexai.generative_models import GenerativeModel, Part
from vertexai.preview.vision_models import ImageGenerationModel

vinit(project=p, location=r)

# Initialize models
g = GenerativeModel("gemini-1.5-pro")  # Primary reasoning model
g1 = GenerativeModel("gemini-1.5-flash")  # Fast model for bulk operations
ig = ImageGenerationModel.from_pretrained("imagegeneration@006")  # Image generation

print("✓ Gemini 1.5 Pro loaded")
print("✓ Gemini 1.5 Flash loaded")
print("✓ Image Generation model loaded")

## 4. ShadowTagAi Core Prompt Templates

In [ ]:
%%python
import json

pt = {
    "sys_shadowtagai_core": "You are shadowtagai—an orchestration and analysis engine. Output JSON or code. Be concise.",
    "sys_swiper": "You are shadowtagai-swiper. Optimize geo-beacon commerce films. Output plans and JSON recipes.",
    "sys_verdict": "You are shadowtagai-verdict. Enforce task flow with locks/escrows and time escalations.",
    "sys_vcm": "You are shadowtagai-vc-mirror. Extract investor theses and generate tailored pitch copy.",
    "sys_geos": "You are shadowtagai-geos. Summarize geo/polygon events and economic triggers.",
    "sys_odor": "You are shadowtagai-odor. Model airflow/odor/CBRN scrubbing with constraints.",
    "sys_tokable": "You are shadowtagai-tokable. Create emotion-first creator flows.",
    "sys_mcarlo": "You are shadowtagai-mcarlo. Run compact Monte Carlo valuations (JSON in/out).",
}

# Save to file
with open("/mnt/data/shadowtagai_prompts_core.json", "w") as f:
    json.dump(pt, f, ensure_ascii=False, indent=2)

print(f"✓ {len(pt)} prompt templates loaded")
print(f"  Agents: {', '.join([k.replace('sys_shadowtagai_', '').replace('sys_', '') for k in pt])}")

## 5. OCR & Vision Tools

In [ ]:
%%python
import base64
import json

from google.cloud import vision as gv

cl = gv.ImageAnnotatorClient()


def ocr_b64(b64):
    """OCR from base64 encoded image"""
    img = gv.Image(content=base64.b64decode(b64))
    r = cl.document_text_detection(image=img)
    t = (
        r.full_text_annotation.text
        if r.full_text_annotation and r.full_text_annotation.text
        else ""
    )
    return t.strip()


def ocr_file(fp):
    """OCR from local file path"""
    with open(fp, "rb") as f:
        bb = base64.b64encode(f.read()).decode()
    return ocr_b64(bb)


def ocr_gcs(uri):
    """OCR from GCS URI (gs://bucket/path)"""
    img = gv.Image()
    img.source.image_uri = uri
    r = cl.document_text_detection(image=img)
    t = (
        r.full_text_annotation.text
        if r.full_text_annotation and r.full_text_annotation.text
        else ""
    )
    return t.strip()


print("✓ OCR tools loaded (b64, file, GCS)")

## 6. OCR + Summarization Pipeline

In [ ]:
%%python


def sum_text(
    txt,
    sys="Summarize salient facts, numeric claims, and calls-to-action. Output JSON with keys: facts, metrics, actions.",
):
    """Summarize text using Gemini Pro"""
    m = g.generate_content(
        [
            {"role": "system", "parts": [sys]},
            {"role": "user", "parts": [txt[:100000]]},  # Max 100K chars
        ],
        generation_config={"temperature": 0.2},
    )
    return m.text


def ocr_and_sum_file(fp):
    """OCR + summarize from local file"""
    return sum_text(ocr_file(fp))


def ocr_and_sum_gcs(uri):
    """OCR + summarize from GCS"""
    return sum_text(ocr_gcs(uri))


print("✓ OCR + summarization pipeline ready")

## 7. JSON & Statistics Utilities

In [ ]:
%%python
import json
import random
import statistics as st


def j(x):
    """Compact JSON serialization"""
    return json.dumps(x, ensure_ascii=False, separators=(",", ":"))


def prc(x, p):
    """Calculate percentile"""
    x = sorted(x)
    i = int((len(x) - 1) * p)
    return x[i]


print("✓ JSON & stats utilities loaded")

## 8. Monte Carlo Valuation Engine

In [ ]:
%%python


def mcarlo_rev(n, base, sd, yrs=5, gr=0.6):
    """Monte Carlo revenue simulation

    Args:
        n: Number of simulations
        base: Base revenue (current)
        sd: Standard deviation
        yrs: Years to project
        gr: Growth rate (CAGR)

    Returns:
        List of projected revenues
    """
    r = []
    for _ in range(n):
        b = max(0, random.gauss(base, sd))
        v = b * ((1 + gr) ** yrs)
        r.append(v)
    return r


def mcarlo_val(n, rev_mult, rev_samples):
    """Calculate valuation from revenue samples

    Args:
        n: Number of samples
        rev_mult: Revenue multiple (e.g., 12×)
        rev_samples: List of revenue projections

    Returns:
        Dict with mean, p10, p50, p90, max
    """
    out = [max(0, rev_mult * rv) for rv in rev_samples]
    return {
        "mean": st.mean(out),
        "p10": prc(out, 0.1),
        "p50": prc(out, 0.5),
        "p90": prc(out, 0.9),
        "max": max(out),
    }


def mcarlo_bundle(cfg):
    """Run Monte Carlo for multiple components

    Args:
        cfg: Dict of component configs

    Returns:
        Dict with components and sum_mean
    """
    vals = []
    comps = {}

    for k, v in cfg.items():
        rs = mcarlo_rev(v["n"], v["base"], v["sd"], v.get("yrs", 5), v.get("gr", 0.6))
        vv = mcarlo_val(v["n"], v["mult"], rs)
        comps[k] = vv
        vals.append(vv["mean"])

    t = sum(vals)
    return {"components": comps, "sum_mean": t}


print("✓ Monte Carlo valuation engine ready")

## 9. ShadowTagAi Bundle Valuation Scenarios

In [ ]:
%%python
cfg = {
    "shadowtagai_core": {"n": 8000, "base": 120e6, "sd": 50e6, "gr": 0.55, "mult": 12},
    "shadowtagai_swiper": {"n": 8000, "base": 200e6, "sd": 90e6, "gr": 0.6, "mult": 13},
    "shadowtagai_geos": {"n": 8000, "base": 30e6, "sd": 20e6, "gr": 0.5, "mult": 22},
    "shadowtagai_odor": {"n": 8000, "base": 120e6, "sd": 60e6, "gr": 0.5, "mult": 10},
    "shadowtagai_verdict": {"n": 8000, "base": 120e6, "sd": 60e6, "gr": 0.55, "mult": 11},
    "shadowtagai_vcm": {"n": 8000, "base": 40e6, "sd": 20e6, "gr": 0.6, "mult": 11},
    "shadowtagai_tokable": {"n": 8000, "base": 80e6, "sd": 40e6, "gr": 0.55, "mult": 11},
}

res = mcarlo_bundle(cfg)

# Save results
with open("/mnt/data/shadowtagai_valuation.json", "w") as f:
    f.write(j(res))

print("✓ ShadowTagAi bundle valuation complete")
print(f"  Total valuation (mean): ${res['sum_mean'] / 1e9:.2f}B")
print(f"  Components: {len(res['components'])}")

## 10. ShadowTagAi Swiper (Geo-Beacon Commerce)

In [ ]:
%%python


def swiper_plan(q):
    """Generate swiper campaign plan

    Args:
        q: Query describing the campaign

    Returns:
        JSON with {wedge, geo, content, cpms, revshare, visualize}
    """
    s = "You are shadowtagai-swiper. Return JSON {wedge,geo,content,cpms,revshare,visualize}."
    m = g1.generate_content(
        [{"role": "system", "parts": [s]}, {"role": "user", "parts": [q]}],
        generation_config={"temperature": 0.3},
    )
    return m.text


def swiper_visualize(img_b64, gear):
    """Visualize gear placement in scene

    Args:
        img_b64: Base64 encoded image
        gear: Gear/product to place

    Returns:
        JSON with placements and style
    """
    t = f"Blend {gear} into scene, return JSON placements and style."
    m = g1.generate_content(
        [
            {"role": "system", "parts": ["Return JSON only."]},
            {
                "role": "user",
                "parts": [
                    Part.from_data(
                        mime_type="image/jpeg", data=__import__("base64").b64decode(img_b64)
                    ),
                    t,
                ],
            },
        ],
        generation_config={"temperature": 0.2},
    )
    return m.text


print("✓ ShadowTagAi Swiper loaded")

## 11. ShadowTagAi Verdict (Task Flow Enforcement)

In [ ]:
%%python


class Verdict:
    """Task flow enforcement with locks/escrows and time escalations"""

    def __init__(self):
        self.q = []  # Task queue

    def add(self, t, dl, prio):
        """Add task with deadline and priority

        Args:
            t: Task description
            dl: Deadline (unix timestamp)
            prio: Priority (higher = more important)
        """
        self.q.append({"t": t, "dl": dl, "p": prio, "k": 0, "d": False})

    def tick(self, now):
        """Update urgency based on time

        Args:
            now: Current time (unix timestamp)
        """
        for i in self.q:
            if i["d"]:  # Skip done tasks
                continue
            if now >= i["dl"]:
                i["k"] = 2  # Overdue
            elif (i["dl"] - now) <= 900:  # 15 min warning
                i["k"] = 1  # Urgent

    def next(self):
        """Get next task (highest urgency/priority)

        Returns:
            Task dict or None
        """
        u = [x for x in self.q if not x["d"]]
        u.sort(key=lambda x: (-x["k"], -x["p"]))
        return u[0] if u else None

    def done(self, t):
        """Mark task as done

        Args:
            t: Task description

        Returns:
            True if found and marked, False otherwise
        """
        for i in self.q:
            if i["t"] == t:
                i["d"] = True
                return True
        return False


# Global verdict instance
V = Verdict()

print("✓ ShadowTagAi Verdict loaded")

## 12. ShadowTagAi VC Mirror (Investor Thesis Extraction)

In [ ]:
%%python


def vcmirror(profile_txt, company):
    """Extract investor thesis and generate tailored pitch

    Args:
        profile_txt: Investor profile/background text
        company: Company description

    Returns:
        JSON with {thesis, fit, angles, email_open, deck_outline}
    """
    sys = "Extract investor thesis and produce pitch copy. JSON keys: thesis,fit,angles,email_open,deck_outline."
    m = g.generate_content(
        [{"role": "system", "parts": [sys]}, {"role": "user", "parts": [profile_txt, company]}],
        generation_config={"temperature": 0.2},
    )
    return m.text


print("✓ ShadowTagAi VC Mirror loaded")

## 13. ShadowTagAi Geos (Geo-Economic Analysis)

In [ ]:
%%python


def geos_skim(txt):
    """Summarize geo triggers, actors, capital flow

    Args:
        txt: Input text (news, reports, etc.)

    Returns:
        JSON summary
    """
    sys = "Summarize geo triggers, actors, capital flow, compliance. JSON only."
    m = g1.generate_content(
        [{"role": "system", "parts": [sys]}, {"role": "user", "parts": [txt]}],
        generation_config={"temperature": 0.2},
    )
    return m.text


print("✓ ShadowTagAi Geos loaded")

## 14. ShadowTagAi Odor (Airflow/CBRN Modeling)

In [ ]:
%%python
import numpy as np


def odor_sim(n=128, src=None, k=0.92, fx=0.02):
    """Simulate odor/airflow diffusion

    Args:
        n: Grid size (n×n)
        src: List of (x, y, strength) sources
        k: Decay constant
        fx: Diffusion factor

    Returns:
        2D numpy array of concentrations
    """
    if src is None:
        src = [(64, 64, 1.0)]
    f = np.zeros((n, n), float)

    for _ in range(256):  # Iterations
        nf = np.copy(f)
        # Diffusion step
        nf[1:-1, 1:-1] = f[1:-1, 1:-1] * k + fx * (
            f[:-2, 1:-1] + f[2:, 1:-1] + f[1:-1, :-2] + f[1:-1, 2:]
        )
        # Add sources
        for x, y, s in src:
            nf[x % n, y % n] += s
        f = nf

    return f


def odor_score(f, mask=None):
    """Calculate odor score (average concentration)

    Args:
        f: Concentration field (2D array)
        mask: Optional mask (2D array, 1=measure, 0=ignore)

    Returns:
        Average concentration
    """
    if mask is None:
        return float(f.mean())
    return float((f * mask).mean())


print("✓ ShadowTagAi Odor loaded")

## 15. ShadowTagAi Tokable (Emotion-First Creator Scripts)

In [ ]:
%%python


def tokable_script(theme, persona):
    """Generate emotion-first creator script

    Args:
        theme: Content theme
        persona: Creator persona/voice

    Returns:
        JSON with 45-90s script, 3 beats, interactive cues
    """
    sys = "Write a 45-90s emotion-first script with 3 beats and optional interactive branch cues. JSON only."
    m = g1.generate_content(
        [{"role": "system", "parts": [sys]}, {"role": "user", "parts": [f"{theme} | {persona}"]}],
        generation_config={"temperature": 0.55},
    )
    return m.text


print("✓ ShadowTagAi Tokable loaded")

## 16. Export Prompt Templates for Studio

In [ ]:
%%python
import json

tpl = [
    {"id": "shadowtagai_core", "role": "system", "content": pt["sys_shadowtagai_core"]},
    {"id": "shadowtagai_swiper", "role": "system", "content": pt["sys_swiper"]},
    {"id": "shadowtagai_verdict", "role": "system", "content": pt["sys_verdict"]},
    {"id": "shadowtagai_vcm", "role": "system", "content": pt["sys_vcm"]},
    {"id": "shadowtagai_geos", "role": "system", "content": pt["sys_geos"]},
    {"id": "shadowtagai_odor", "role": "system", "content": pt["sys_odor"]},
    {"id": "shadowtagai_tokable", "role": "system", "content": pt["sys_tokable"]},
    {"id": "shadowtagai_mcarlo", "role": "system", "content": pt["sys_mcarlo"]},
]

with open("/mnt/data/shadowtagai_studio_templates.json", "w") as f:
    f.write(j(tpl))

print("✓ Studio templates exported")
print(f"  Templates: {len(tpl)}")

## 17. Initialization Complete

In [ ]:
%%python
print("\n" + "=" * 60)
print("  SHADOWTAGAI VERTEX INITIALIZATION COMPLETE")
print("=" * 60)
print("\n✓ Gemini models loaded (Pro, Flash, ImageGen)")
print("✓ 8 ShadowTagAi agents ready:")
print("  • swiper (geo-beacon commerce)")
print("  • verdict (task flow enforcement)")
print("  • vcm (investor thesis extraction)")
print("  • geos (geo-economic analysis)")
print("  • odor (airflow/CBRN modeling)")
print("  • tokable (emotion-first creator scripts)")
print("  • mcarlo (Monte Carlo valuations)")
print("  • core (orchestration engine)")
print("\n✓ OCR & vision tools ready")
print("✓ Monte Carlo valuation engine ready")
print("✓ GCS helpers ready (up, dl)")
print("\nReady for production use.")
print("=" * 60)